# UD7.01. El proceso a mano: el punto de referencia

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloque 2 de los apuntes · Criterios **4.b** y **4.c**

---

Antes de decidir si automatizar algo hay que saber qué cuesta hoy. No «más o menos»:
con números, y con los cinco que hacen falta.

Este cuaderno mide el buzón de TechStore tal como funciona ahora, con una persona
leyendo, clasificando y encaminando 2.400 mensajes al año. Al final tienes que poder
contestar estas cuatro, que salen en el cuestionario y que hacen falta para la P7.1:

1. ¿Cuánto cuesta al año el proceso, **contando los errores**?
2. ¿Cuánto de lo que espera un cliente es trabajo, cuánto es cola y cuánto es que la
   oficina está cerrada?
3. ¿Con qué frecuencia se equivoca la persona, y **en qué se equivoca**?
4. ¿Qué error del proceso es el caro, y cuánto de caro?

> **Sobre los datos.** El buzón es sintético y lo genera `genera_datos_ud7.py` con
> semilla fija: todo el mundo mide lo mismo. Tiene una columna que en una empresa real
> **no existiría**, `categoria`, que es la verdad. Está aquí para poder medir contra
> qué. Lo que una empresa tiene es `categoria_asignada`: lo que puso la persona.

In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)

# Solo en Google Colab: descarga los ficheros de datos de la unidad. En local ya están
# en recursos/datos/ y esta celda no hace nada.
#
# Ojo con la ruta: en el espejo público la unidad se llama UD7, no
# UD7_automatizacion_negocio, porque sincroniza_notebooks.py acorta el nombre.
BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD7/datos/")
os.makedirs("datos", exist_ok=True)
for fichero in ["buzon_techstore.csv", "procesos_techstore.csv"]:
    ruta = os.path.join("datos", fichero)
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(BASE + fichero, ruta)
        print("descargado:", fichero)
    else:
        print("ya está:", fichero)

buzon = pd.read_csv(os.path.join("datos", "buzon_techstore.csv"),
                    parse_dates=["fecha_hora"])
print()
print(buzon.shape)
buzon.head(3)

## 1. Las cinco cifras de un proceso

| Cifra | Qué es |
|---|---|
| **Volumen** | Casos por unidad de tiempo |
| **Tiempo de toque** | Lo que tarda una persona en tratar uno |
| **Tiempo de ciclo** | Desde que entra hasta que se resuelve |
| **Tasa de error** | Con qué frecuencia se decide mal |
| **Coste del error** | Lo que cuesta cada decisión mal tomada |

Las tres primeras son de manual. Las dos últimas son las que deciden, y son las que
casi nadie mide.

Empezamos por las dos fáciles. Y por un supuesto que hay que declarar antes de usarlo:

In [ ]:
# Coste de empresa de una hora de la persona de atención al cliente: salario bruto de
# convenio más la cotización a cargo de la empresa, dividido entre las horas anuales.
# Es una cifra de trabajo, no un dato de contabilidad de TechStore. Todo lo que salga
# de aquí depende de ella, así que va escrita y con nombre.
COSTE_HORA = 18.50

minutos = buzon["minutos_manual"].sum()
horas = minutos / 60
coste_personal = horas * COSTE_HORA

print(f"Mensajes al año           {len(buzon):>10,}")
print(f"Tiempo de toque medio     {buzon['minutos_manual'].mean():>10.1f} min")
print(f"Minutos al año            {minutos:>10,.0f}")
print(f"Horas al año              {horas:>10,.1f}")
print(f"Coste de personal al año  {coste_personal:>10,.0f} EUR")
print(f"Coste por mensaje         {coste_personal / len(buzon):>10.2f} EUR")
print()
# Una jornada anual de convenio son unas 1.758 horas.
print(f"Esas horas son {horas / 1758:.2f} de una persona a jornada completa.")

> **Primera conclusión, y es incómoda para el proyecto:** el buzón entero ocupa **menos
> de una quinta parte de una persona**. Cualquier promesa de «ahorrar un empleado»
> automatizando esto es falsa antes de empezar, y se descarta con una división.
>
> Medir el proceso sirve, muy a menudo, para tirar a la basura la justificación con la
> que venía el proyecto. Eso no es un mal resultado: es el resultado.

Veamos de qué se compone ese tiempo.

In [ ]:
resumen = (buzon.groupby("categoria")
           .agg(mensajes=("mensaje_id", "size"),
                minutos_medios=("minutos_manual", "mean"),
                minutos_totales=("minutos_manual", "sum"))
           .sort_values("minutos_totales", ascending=False))
resumen["% de los mensajes"] = (resumen["mensajes"] / len(buzon) * 100).round(1)
resumen["% del tiempo"] = (resumen["minutos_totales"] / minutos * 100).round(1)
resumen["coste_ano"] = (resumen["minutos_totales"] / 60 * COSTE_HORA).round(0)
print(resumen)
print()
print("Fíjate en las dos columnas de porcentaje: no coinciden. Las garantías son el")
print("18 % de los mensajes y se llevan casi un tercio del tiempo, porque cada una")
print("cuesta casi el triple que una consulta de envío.")

## 2. El tiempo de ciclo, que es lo que nota el cliente

El tiempo de toque es lo que tarda la persona. El **tiempo de ciclo** es lo que espera
el cliente, desde que escribe hasta que alguien le contesta. No se parecen en nada, y
confundirlos es el error más frecuente al presentar una automatización.

Para calcularlo hay que simular la cola: una persona, cinco horas al día, de nueve a
dos, de lunes a viernes, por orden de llegada.

In [ ]:
HORAS_DIA = 5.0
VENTANA = (9, 14)          # de 9:00 a 14:00


def simula_cola(buzon):
    """Devuelve, para cada mensaje, el instante en que se termina de tratar.

    No es teoría de colas: es un bucle. Se hace así porque lo que hay que ver es que
    el tiempo de ciclo NO es el tiempo de trabajo, y un bucle lo enseña mejor que una
    fórmula.
    """
    abre, cierra = VENTANA
    reloj = None
    fin = []
    for llegada, minutos_caso in zip(buzon["fecha_hora"], buzon["minutos_manual"]):
        reloj = llegada if reloj is None else max(reloj, llegada)
        restante = float(minutos_caso)
        while restante > 0:
            if reloj.weekday() >= 5 or reloj.hour >= cierra:
                dia = reloj.normalize() + pd.Timedelta(days=1)
                while dia.weekday() >= 5:
                    dia += pd.Timedelta(days=1)
                reloj = dia + pd.Timedelta(hours=abre)
                continue
            if reloj.hour < abre:
                reloj = reloj.normalize() + pd.Timedelta(hours=abre)
                continue
            cierre_hoy = reloj.normalize() + pd.Timedelta(hours=cierra)
            usado = min((cierre_hoy - reloj).total_seconds() / 60, restante)
            reloj += pd.Timedelta(minutes=usado)
            restante -= usado
        fin.append(reloj)
    return pd.Series(fin, index=buzon.index)


fin = simula_cola(buzon)
ciclo_h = (fin - buzon["fecha_hora"]).dt.total_seconds() / 3600

print(f"Tiempo de toque medio          {buzon['minutos_manual'].mean() / 60:>8.2f} h")
print(f"Tiempo de ciclo mediano        {ciclo_h.median():>8.1f} h")
print(f"Tiempo de ciclo, percentil 90  {ciclo_h.quantile(0.90):>8.1f} h")
print(f"Tiempo de ciclo máximo         {ciclo_h.max():>8.1f} h")
print()
print(f"El ciclo mediano es {ciclo_h.median() / (buzon['minutos_manual'].mean() / 60):.0f} "
      f"veces el tiempo de trabajo.")

### ¿Esperando a qué?

Aquí está la pregunta que separa un análisis de una ocurrencia. Que el mensaje espere
no dice nada; **a qué espera** lo dice todo, porque las dos esperas se arreglan con
cosas distintas:

- La espera **por horario** se ataca con algo que conteste a las tres de la mañana.
- La espera **por cola** se ataca con más manos o con menos trabajo por mensaje.

Confundirlas es el error típico de quien vende automatización. Vamos a separarlas.

> **Animación interactiva:** [Esperando a que abran](https://rafasalaesteve.github.io/IABD-PIA/UD7_automatizacion_negocio/animaciones/01_esperando_a_que_abran.html). Permite recorrer una semana del buzón mensaje a mensaje y ver cuánto de cada tiempo de ciclo es oficina cerrada, cuánto es cola y cuánto es trabajo.

In [ ]:
def descompone_la_espera(buzon, fin):
    """Separa los minutos laborables (cola) de los no laborables (oficina cerrada)."""
    abre, cierra = VENTANA
    inicio = fin - pd.to_timedelta(buzon["minutos_manual"], unit="m")
    por_horario, por_cola = [], []
    for llegada, arranque in zip(buzon["fecha_hora"], inicio):
        laborables, cursor = 0.0, llegada
        while cursor < arranque:
            ini_dia = cursor.normalize() + pd.Timedelta(hours=abre)
            fin_dia = cursor.normalize() + pd.Timedelta(hours=cierra)
            if cursor.weekday() < 5 and ini_dia <= cursor < fin_dia:
                tramo = min(fin_dia, arranque) - cursor
                laborables += tramo.total_seconds() / 3600
                cursor = min(fin_dia, arranque)
            else:
                siguiente = cursor.normalize() + pd.Timedelta(days=1) + pd.Timedelta(hours=abre)
                if cursor < ini_dia and cursor.weekday() < 5:
                    siguiente = ini_dia
                cursor = min(siguiente, arranque)
        total = (arranque - llegada).total_seconds() / 3600
        por_cola.append(laborables)
        por_horario.append(max(total - laborables, 0.0))
    return pd.Series(por_horario, index=buzon.index), pd.Series(por_cola, index=buzon.index)


por_horario, por_cola = descompone_la_espera(buzon, fin)
toque_h = buzon["minutos_manual"] / 60
total = ciclo_h.sum()

print(f"Del tiempo de ciclo total del año:")
print(f"  trabajo de verdad   {toque_h.sum() / total:>7.1%}")
print(f"  oficina cerrada     {por_horario.sum() / total:>7.1%}")
print(f"  cola                {por_cola.sum() / total:>7.1%}")
print()
ocupacion = toque_h.sum() / (261 * HORAS_DIA)
print(f"Ocupación de la persona: {ocupacion:.1%} de las {HORAS_DIA:.0f} h/día dedicadas")

> **Esta es la cifra que cambia la conversación.** El mensaje casi no espera detrás de
> otros: espera a que abran. Con la persona ocupada un cuarto de su tiempo, **poner a
> otra persona no arreglaría casi nada**.
>
> Y de ahí sale una distinción que hay que tener clara desde el primer día del
> proyecto:
>
> - Si el objetivo es **responder más rápido**, lo que hace falta es cualquier cosa que
>   conteste fuera del horario de oficina. No hace falta un modelo.
> - Si el objetivo es **gastar menos**, hay que mirar los minutos y los errores.
>
> Son dos proyectos distintos, con tecnologías distintas y con justificaciones
> distintas. Confundirlos es el error más caro de esta unidad.

Vamos a verlo también por hora de llegada, porque explica por dónde entra la espera.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))

por_hora = buzon.groupby(buzon["fecha_hora"].dt.hour).size()
ejes[0].bar(por_hora.index, por_hora.values, color="#4878A8")
ejes[0].axvspan(VENTANA[0], VENTANA[1], color="#8ECF7F", alpha=0.30,
                label="horario de atención")
ejes[0].set_title("Mensajes que entran, por hora")
ejes[0].set_xlabel("hora de entrada")
ejes[0].set_ylabel("mensajes al año")
ejes[0].legend()

ejes[1].hist(ciclo_h, bins=40, color="#4878A8")
ejes[1].axvline(ciclo_h.median(), color="#B4453C", linewidth=2,
                label=f"mediana {ciclo_h.median():.1f} h")
ejes[1].axvline(toque_h.mean(), color="#3B7A57", linewidth=2,
                label=f"trabajo real {toque_h.mean():.2f} h")
ejes[1].set_title("Tiempo de ciclo")
ejes[1].set_xlabel("horas")
ejes[1].set_ylabel("mensajes")
ejes[1].legend()

plt.tight_layout()
plt.show()

fuera = (~buzon["fecha_hora"].dt.hour.between(VENTANA[0], VENTANA[1] - 1)
         | (buzon["fecha_hora"].dt.weekday >= 5)).mean()
print(f"El {fuera:.0%} de los mensajes entra fuera del horario de atención.")

## 3. La tasa de error: la persona no acierta el 100 %

Aquí está la trampa más común al presentar una automatización: compararla con un humano
imaginario que no se equivoca nunca. Ese humano no existe, y si se usa como punto de
referencia **cualquier automatización parece un retroceso**.

La columna `categoria_asignada` es lo que puso la persona. La columna `categoria` es lo
que era. Vamos a ver en cuánto se parecen.

In [ ]:
acierta = buzon["categoria"] == buzon["categoria_asignada"]
print(f"Acierto de la persona      {acierta.mean():>8.1%}")
print(f"Mensajes mal encaminados   {(~acierta).sum():>8,}  al año")
print(f"                           {(~acierta).sum() / 261:>8.1f}  al día laborable")
print()

print("Acierto por hora de entrada:")
por_hora = buzon.groupby(buzon["fecha_hora"].dt.hour).apply(
    lambda g: (g["categoria"] == g["categoria_asignada"]).mean(), include_groups=False)
for hora, valor in por_hora.items():
    barra = "#" * int(valor * 50)
    print(f"  {hora:>2}:00  {valor:>6.1%}  {barra}")

El acierto cae a partir de las cinco de la tarde. No es incompetencia: es lo que le pasa
a cualquiera clasificando mensajes ambiguos al final del día. Es una característica del
proceso manual, no de la persona, y por tanto **no se arregla contratando a otra**.

Ahora, dónde se equivoca. Una matriz de confusión con las filas en lo que era y las
columnas en lo que puso:

In [ ]:
confusion = pd.crosstab(buzon["categoria"], buzon["categoria_asignada"],
                        rownames=["era"], colnames=["puso"])
print(confusion)
print()

errores = buzon[~acierta]
print("Las cinco confusiones más frecuentes:")
pares = (errores.groupby(["categoria", "categoria_asignada"]).size()
         .sort_values(ascending=False).head(5))
for (era, puso), n in pares.items():
    print(f"  {era:>11} -> {puso:<11} {n:>4} veces")

La confusión dominante es **garantía contra devolución**, en las dos direcciones. No es
casual: son los mensajes que el conjunto marca como `frontera`, y su diferencia no está
en el texto sino en cuánto hace que se compró el aparato. Catorce días o menos es una
devolución —el plazo legal de desistimiento del artículo 102 del texto refundido de la
Ley General para la Defensa de los Consumidores y Usuarios—; más de catorce días es una
garantía.

Vamos a comprobar que es eso y no otra cosa.

In [ ]:
for grupo, nombre in [(buzon["frontera"] == 1, "de frontera"),
                      (buzon["confuso"] == 1, "confusos"),
                      (buzon["ambiguo"] == 1, "con dos intenciones"),
                      ((buzon["frontera"] == 0) & (buzon["confuso"] == 0)
                       & (buzon["ambiguo"] == 0), "nítidos")]:
    sub = buzon[grupo]
    exactitud = (sub["categoria"] == sub["categoria_asignada"]).mean()
    print(f"{nombre:>20}: {len(sub):>5} mensajes ({grupo.mean():>5.1%}), "
          f"la persona acierta el {exactitud:.1%}")

print()
print("Las columnas `frontera`, `confuso` y `ambiguo` NO existirían en una empresa.")
print("Están en el conjunto para poder medir dónde está la dificultad. No se pueden")
print("usar como características de ningún modelo: eso sería hacer trampa.")

## 4. El coste del error, que no es uno solo

Y aquí está el salto conceptual del cuaderno.

> **Un error no cuesta «un error»: cuesta lo que cuesta arreglar ese error concreto.**

Encaminar mal un mensaje de publicidad cuesta unos minutos. Encaminar mal una garantía
cuesta un escalado, una sustitución y el riesgo de una reclamación de consumo. No son
comparables, y sumarlos como si lo fueran es lo que hace que una automatización parezca
buena cuando no lo es.

Los precios de abajo son **órdenes de magnitud justificados**, no datos de contabilidad.
Eso está bien, con una condición que esta unidad exige en todas las entregas: **un coste
sin justificación no puntúa**.

In [ ]:
COSTE_ERROR = {
    "spam": 1.50,        # unos minutos perdidos tratando publicidad
    "envio": 8.00,       # el cliente vuelve a escribir: se trata dos veces
    "devolucion": 18.00,  # se pasa el plazo de 14 días y hay que aceptarla igual
    "factura": 25.00,    # reclamación, rehacer el documento y a veces la gestoría
    "producto": 35.00,   # se pierde la venta y lo que costó captar a esa persona
    "garantia": 120.00,  # escalado, sustitución y riesgo de reclamación de consumo
}
# Y una asimetría que importa más que los precios: si un mensaje acaba en la bandeja de
# publicidad, no lo vuelve a mirar NADIE. Ese error cuesta 2,5 veces más.
PENALIZACION_A_SPAM = 2.5


def coste_de_los_errores(datos, prediccion):
    verdad = datos["categoria"].to_numpy()
    fallos = prediccion != verdad
    return sum(COSTE_ERROR[real] * (PENALIZACION_A_SPAM if puesto == "spam" else 1.0)
               for real, puesto in zip(verdad[fallos], prediccion[fallos]))


coste_errores = coste_de_los_errores(buzon, buzon["categoria_asignada"].to_numpy())

print(f"Coste de personal al año   {coste_personal:>10,.0f} EUR")
print(f"Coste de los errores       {coste_errores:>10,.0f} EUR")
print(f"                           {'-' * 10}")
print(f"COSTE TOTAL DEL PROCESO    {coste_personal + coste_errores:>10,.0f} EUR al año")
print()
print(f"Por mensaje: {(coste_personal + coste_errores) / len(buzon):.2f} EUR, de los")
print(f"cuales {coste_personal / len(buzon):.2f} son tiempo y "
      f"{coste_errores / len(buzon):.2f} son errores.")

> **Segunda conclusión, y es la que justifica el proyecto:** en este proceso **el dinero
> no está en el tiempo, está en los errores**. Los errores cuestan el doble que las
> horas.
>
> Quien defienda la automatización del buzón por el ahorro de horas está defendiendo el
> tercio menos importante del caso. Y quien no haya puesto precio a los errores no sabe
> qué está decidiendo.

Veamos de dónde sale ese dinero, porque tampoco está repartido.

In [ ]:
# El coste se calcula mensaje a mensaje y NO como errores x precio, porque los que
# acaban en la papelera valen 2,5 veces más. Si se calcula con la multiplicación
# rápida, la tabla no suma lo mismo que el total de arriba y no cuadra nada.
coste_fila = errores.apply(
    lambda f: COSTE_ERROR[f["categoria"]]
    * (PENALIZACION_A_SPAM if f["categoria_asignada"] == "spam" else 1.0), axis=1)

detalle = (errores.assign(coste=coste_fila).groupby("categoria")
           .agg(errores=("mensaje_id", "size"), coste_total=("coste", "sum"))
           .assign(coste_unitario=lambda d: d.index.map(COSTE_ERROR))
           .sort_values("coste_total", ascending=False))
detalle["% del coste"] = (detalle["coste_total"] / detalle["coste_total"].sum() * 100).round(1)
print(detalle)
assert abs(detalle["coste_total"].sum() - coste_errores) < 0.01, "el desglose no cuadra"
print()
garantias_perdidas = int(((buzon["categoria"] == "garantia") & ~acierta).sum())
a_papelera = int(((buzon["categoria"] != "spam")
                  & (buzon["categoria_asignada"] == "spam")).sum())
print(f"Garantías mal encaminadas al año: {garantias_perdidas}")
print(f"Mensajes que no eran publicidad y acabaron en la papelera: {a_papelera}")

**Sesenta y dos garantías al año mal encaminadas.** Sesenta y dos clientes con un aparato
roto cuya reclamación va al equipo que no es. Ese es el punto de referencia real del
proceso, y es mucho peor de lo que nadie habría supuesto antes de medirlo.

Guárdalo, porque es contra esto contra lo que hay que comparar cualquier automatización
en el cuaderno `UD7_03`. No contra el 100 %.

## 5. Lo que hay que llevarse de este cuaderno

In [ ]:
print(f"""
PUNTO DE REFERENCIA DEL BUZÓN DE TECHSTORE
==========================================
Volumen                         {len(buzon):>10,} mensajes al año
Tiempo de toque medio           {buzon['minutos_manual'].mean():>10.1f} min
Tiempo de ciclo mediano         {ciclo_h.median():>10.1f} h
  de ese ciclo, trabajo         {toque_h.sum() / total:>10.1%}
  de ese ciclo, oficina cerrada {por_horario.sum() / total:>10.1%}
  de ese ciclo, cola            {por_cola.sum() / total:>10.1%}
Acierto de la persona           {acierta.mean():>10.1%}
Errores al año                  {(~acierta).sum():>10,}
  de ellos, garantías           {garantias_perdidas:>10}

Coste de personal               {coste_personal:>10,.0f} EUR/año
Coste de los errores            {coste_errores:>10,.0f} EUR/año
COSTE TOTAL                     {coste_personal + coste_errores:>10,.0f} EUR/año
""")

Tres frases para llevarse:

1. **El proceso cuesta 18.805 euros al año, y dos tercios son errores.** El tiempo es lo
   de menos.
2. **El 97 % de lo que espera el cliente es que la oficina está cerrada.** No hay cola
   que resolver.
3. **La persona acierta el 86,8 %, y falla sobre todo en garantía contra devolución**,
   que es una distinción que no está en el texto del mensaje.

La tercera es la que abre el cuaderno siguiente: si la diferencia no está en el texto,
¿de qué le sirve a un modelo leer el texto?

---

### Para la P7.1

Este cuaderno da el punto de referencia; `UD7_02` da los modelos. La práctica te va a
pedir las dos cosas en la misma tabla, y con **tus** números: si cambias `COSTE_HORA` o
los precios de `COSTE_ERROR`, cambian todos los resultados de la unidad, y por eso hay
que declararlos junto a cualquier cifra que salga de ellos.